# Limpeza de dados - Steam Player Data
Projeto final: Análise de Dados e BI (setor Games)

## 1. Carga dos dados
Baixar o CSV no Kaggle (jackogozaly / Steam Player Data) e enviar o arquivo quando o Colab pedir.

In [ ]:
import re
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
nome_arquivo = list(uploaded.keys())[0]
df_raw = pd.read_csv(nome_arquivo)
df = df_raw.copy()  # df_raw permanece intacto para comparação antes/depois

print(df.shape)
df.head()

## 2. Inspeção inicial

In [ ]:
df.info()
print()
print("Nulos por coluna:")
print(df.isnull().sum())
print()
print("Linhas duplicadas (linha inteira):", df.duplicated().sum())

## 3. Padronização dos nomes das colunas
Nomes em minúsculas e snake_case para evitar erros de digitação nas etapas seguintes.

In [ ]:
def padronizar_coluna(nome):
    nome = nome.strip().lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    return nome.strip("_")

df.columns = [padronizar_coluna(c) for c in df.columns]
print(list(df.columns))

In [ ]:
# Ajustar as chaves conforme a saída da célula anterior, se os nomes forem diferentes
RENOMEAR = {
    "gamename": "game_name",
    "name": "game_name",
}
df = df.rename(columns=RENOMEAR)

COLUNAS_ESPERADAS = ["month_year", "avg_players", "gain", "percent_gain",
                     "peak_players", "url", "date", "game_name"]
faltando = [c for c in COLUNAS_ESPERADAS if c not in df.columns]
print("Colunas faltando:", faltando)

## 4. Tipos e formatação
- `date` convertida para datetime.
- `percent_gain` vem como texto com símbolo `%` e sinal `+` (ex: `+0.05%`); convertida para número.
- Nomes de jogos: remoção de espaços extras (há registros com espaço no fim do nome) e correção de erros de digitação do dataset.

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

for col in ["avg_players", "gain", "peak_players"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["percent_gain"] = (
    df["percent_gain"].astype(str)
    .str.replace("%", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)
df["percent_gain"] = pd.to_numeric(df["percent_gain"], errors="coerce")

# Espaços extras no nome fariam o mesmo jogo ser tratado como dois grupos
com_espaco = (df["game_name"] != df["game_name"].astype(str).str.strip()).sum()
print("Linhas com espaço extra no nome do jogo:", com_espaco)
df["game_name"] = df["game_name"].astype(str).str.strip()

# Erros de digitação presentes no dataset original
CORRIGIR_NOMES = {
    "Forza Horizion 4": "Forza Horizon 4",
    "Need foor Speed Heat": "Need for Speed Heat",
    "Tom Clancy's Rainbow Six Seige": "Tom Clancy's Rainbow Six Siege",
}
df["game_name"] = df["game_name"].replace(CORRIGIR_NOMES)

linhas_bruto = len(df_raw)
jogos_bruto = df["game_name"].nunique()
print("Jogos na base original:", jogos_bruto)
df.dtypes

## 5. Dados sensíveis
A base é pública e agregada por mês. Verificação de colunas e de padrões de dado pessoal (e-mail, CPF, telefone) nas colunas de texto.

In [ ]:
print("Colunas da base:", list(df.columns))

PADROES = {
    "email": r"[\w\.-]+@[\w\.-]+\.\w+",
    "cpf": r"\d{3}\.\d{3}\.\d{3}-\d{2}",
    "telefone": r"\(?\d{2}\)?\s?9?\d{4}-\d{4}",
}

achados = 0
for col in df.select_dtypes(include=["object", "string"]).columns:
    for nome, padrao in PADROES.items():
        qtd = df[col].astype(str).str.contains(padrao, regex=True).sum()
        achados += qtd
        if qtd:
            print(f"Coluna {col}: {qtd} ocorrências de {nome}")

print("Total de ocorrências de dado pessoal:", achados)
# Resultado esperado: 0. Colunas são jogo, data, contagens de jogadores (agregadas) e link público,
# sem identificação de usuário. Nenhuma coluna precisa ser anonimizada ou removida.

## 6. Registros que não são jogos
Itens do ranking que são ferramentas ou apps de teste, não jogos, e distorcem a análise de engajamento. Lista editável.

In [ ]:
NAO_JOGOS = ["Spacewar", "Wallpaper Engine", "Soundpad"]

encontrados = [j for j in NAO_JOGOS if j in set(df["game_name"])]
print("Presentes na base:", encontrados)

antes = len(df)
df = df[~df["game_name"].isin(NAO_JOGOS)]
print("Linhas removidas:", antes - len(df))

## 7. Duplicados
Chave lógica: um registro por jogo por mês (`game_name` + `date`).

In [ ]:
dup_chave = df.duplicated(subset=["game_name", "date"], keep=False).sum()
print("Linhas com chave (jogo, mês) repetida:", dup_chave)

antes = len(df)
df = df.drop_duplicates(subset=["game_name", "date"], keep="first")
print("Removidas:", antes - len(df))

## 8. Meses sem dados (0 jogadores)
Alguns jogos têm meses anteriores ao lançamento com média e pico iguais a zero. Esses registros não representam engajamento e gerariam variação infinita no primeiro mês com jogadores. São removidos antes do cálculo de variação.

In [ ]:
zeros = df[(df["avg_players"] == 0) | (df["peak_players"] == 0)]
print("Linhas com 0 jogadores:", len(zeros))
print(zeros["game_name"].value_counts())

df = df[(df["avg_players"] > 0) & (df["peak_players"] > 0)]
print("Linhas restantes:", len(df))

## 9. Valores nulos e recálculo da variação
Os nulos de `gain` e `percent_gain` no primeiro mês de cada jogo são esperados (não há mês anterior para comparar). Nulos nas colunas principais (`avg_players`, `peak_players`, `date`, `game_name`) inviabilizam a linha e ela é removida. A variação é recalculada por jogo, em ordem cronológica, para garantir consistência.

In [ ]:
print("Nulos antes do tratamento:")
print(df[["avg_players", "peak_players", "date", "game_name", "gain", "percent_gain"]].isnull().sum())

colunas_obrigatorias = ["avg_players", "peak_players", "date", "game_name"]
antes = len(df)
df = df.dropna(subset=colunas_obrigatorias)
print("\nLinhas removidas por nulo em coluna obrigatória:", antes - len(df))

df = df.sort_values(["game_name", "date"]).reset_index(drop=True)
df["gain_calc"] = df.groupby("game_name")["avg_players"].diff()
df["percent_gain_calc"] = df.groupby("game_name")["avg_players"].pct_change() * 100

# Compara o valor original com o recalculado (apenas onde existem os dois)
comparavel = df.dropna(subset=["gain", "gain_calc"])
divergentes = (comparavel["gain"] - comparavel["gain_calc"]).abs() > 0.5
print("Linhas em que gain original diverge do recalculado:", divergentes.sum())

df["gain"] = df["gain_calc"]
df["percent_gain"] = df["percent_gain_calc"]
df = df.drop(columns=["gain_calc", "percent_gain_calc"])

print("Valores infinitos em percent_gain:", np.isinf(df["percent_gain"]).sum())
print("Nulos em gain (primeiro mês de cada jogo):", df["gain"].isnull().sum(), "| jogos:", df["game_name"].nunique())

## 10. Consistência lógica
O pico mensal não pode ser menor que a média mensal. Valores negativos de jogadores são inválidos.

In [ ]:
inconsistentes = df[df["peak_players"] < df["avg_players"]]
print("Pico < média:", len(inconsistentes))

negativos = df[(df["avg_players"] < 0) | (df["peak_players"] < 0)]
print("Valores negativos:", len(negativos))

df = df[(df["peak_players"] >= df["avg_players"]) & (df["avg_players"] >= 0)]
print("Linhas restantes:", len(df))

## 11. Outliers
Método IQR aplicado por jogo, pois a escala de jogadores varia muito entre títulos. Picos de lançamento e eventos são comportamentos reais do negócio, então os outliers são **sinalizados em coluna própria e mantidos**, não removidos.

In [ ]:
def sinalizar_outliers_iqr(grupo):
    q1 = grupo.quantile(0.25)
    q3 = grupo.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    return (grupo < limite_inf) | (grupo > limite_sup)

df["outlier_avg"] = df.groupby("game_name")["avg_players"].transform(sinalizar_outliers_iqr)

print("Outliers sinalizados:", df["outlier_avg"].sum(), "de", len(df))
print(df.groupby("game_name")["outlier_avg"].sum().sort_values(ascending=False).head(10))

## 12. Cobertura temporal por jogo
Jogos com poucos meses de histórico têm pouco valor para análise de tendência de queda.

In [ ]:
cobertura = (
    df.groupby("game_name")
      .agg(meses=("date", "count"),
           primeiro_mes=("date", "min"),
           ultimo_mes=("date", "max"))
      .sort_values("meses")
)
print(cobertura.head(10))
print()
print("Período total:", df["date"].min().date(), "a", df["date"].max().date())
print("Jogos:", df["game_name"].nunique())

In [ ]:
# Mínimo de meses para manter o jogo na análise de tendência
MIN_MESES = 12
jogos_validos = cobertura[cobertura["meses"] >= MIN_MESES].index
removidos = sorted(set(cobertura.index) - set(jogos_validos))
print("Jogos com menos de", MIN_MESES, "meses:", len(removidos))
print(removidos)

df = df[df["game_name"].isin(jogos_validos)].reset_index(drop=True)
print("Linhas finais:", len(df))

## 13. Validação com fonte externa
Conferência manual de uma amostra da base limpa contra o histórico mensal do jogo no SteamCharts (coluna `url`). Registrar abaixo o resultado da conferência.

In [ ]:
amostra = df[(df["game_name"] == "Counter Strike: Global Offensive") &
            (df["date"].between("2020-02-01", "2020-04-01"))]
print("Conferir em:", amostra["url"].iloc[0])
amostra[["date", "avg_players", "peak_players"]]

Resultado da conferência (preencher): ______

## 14. Resumo antes e depois

In [ ]:
resumo = pd.DataFrame({
    "linhas": [linhas_bruto, len(df)],
    "jogos": [jogos_bruto, df["game_name"].nunique()],
}, index=["bruto", "limpo"])
resumo

## 15. Exportação
CSV limpo para as próximas etapas (exploração e dashboard) e dicionário de dados.

In [ ]:
df = df[["game_name", "date", "month_year", "avg_players", "peak_players",
         "gain", "percent_gain", "outlier_avg", "url"]]
df.to_csv("steam_players_limpo.csv", index=False)

dicionario = pd.DataFrame({
    "coluna": ["game_name", "date", "month_year", "avg_players", "peak_players",
               "gain", "percent_gain", "outlier_avg", "url"],
    "tipo": [str(df[c].dtype) for c in df.columns],
    "descricao": [
        "Nome do jogo",
        "Primeiro dia do mês de referência",
        "Mês e ano em texto (origem SteamCharts)",
        "Média mensal de jogadores simultâneos",
        "Pico mensal de jogadores simultâneos",
        "Variação absoluta da média em relação ao mês anterior (recalculada)",
        "Variação percentual da média em relação ao mês anterior (recalculada)",
        "Sinalização de outlier por IQR dentro do jogo (mantido na base)",
        "Link da página do jogo no SteamCharts",
    ],
})
dicionario.to_excel("dicionario_dados.xlsx", index=False)

files.download("steam_players_limpo.csv")
files.download("dicionario_dados.xlsx")